In [3]:

from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")


d:\RAG_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4712.03it/s]


Embedding model loaded successfully.


In [4]:

def ingest_book(
    documents,
    book_id,
    book_title,
    db_path="../chroma_db",
    chunk_size=1000,
    chunk_overlap=200,
    batch_size=64
):
    """
    Split book documents into chunks, generate embeddings,
    and store text, metadata, and embeddings in ChromaDB.
    """

    if not documents:
        raise ValueError("No documents provided.")

    # 1. Split the documents into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_documents(documents)

    if not chunks:
        raise ValueError("No text chunks were created.")

    # 2. Create a persistent ChromaDB database
    client = chromadb.PersistentClient(path=db_path)

    collection = client.get_or_create_collection(
        name="book_chunks",
        metadata={"hnsw:space": "cosine"}
    )

    # 3. Generate IDs unique to each book and chunk
    ids = [
        f"{book_id}_chunk_{i}"
        for i in range(len(chunks))
    ]

    # 4. Store embeddings, text, and metadata in batches
    for start in range(0, len(chunks), batch_size):
        batch_chunks = chunks[start:start + batch_size]
        batch_ids = ids[start:start + batch_size]

        texts = [chunk.page_content for chunk in batch_chunks]

        embeddings = embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

        metadatas = []

        for i, chunk in enumerate(batch_chunks, start=start):
            metadatas.append({
                "book_id": book_id,
                "book_title": book_title,
                "source_file": Path(
                    chunk.metadata.get("source", "unknown")
                ).name,
                "page": int(
                    chunk.metadata.get("page", 0)
                ) + 1,
                "chunk_index": i
            })

        collection.upsert(
            ids=batch_ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas
        )

        print(
            f"Stored {min(start + batch_size, len(chunks))}"
            f"/{len(chunks)} chunks"
        )

    print("\nBook ingestion completed.")
    print("Book ID:", book_id)
    print("Total chunks:", len(chunks))
    print("Database:", Path(db_path).resolve())

    return collection


In [5]:

alice_collection = ingest_book(
    documents=all_pdf_documents,
    book_id="alice_wonderland",
    book_title="Alice's Adventures in Wonderland"
)


Stored 64/245 chunks
Stored 128/245 chunks
Stored 192/245 chunks
Stored 245/245 chunks

Book ingestion completed.
Book ID: alice_wonderland
Total chunks: 245
Database: D:\RAG_project\chroma_db


In [6]:

query = "Why does Alice follow the White Rabbit?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
).tolist()

results = alice_collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"book_id": "alice_wonderland"},
    include=["documents", "metadatas", "distances"]
)

for i in range(len(results["documents"][0])):
    metadata = results["metadatas"][0][i]

    print("-" * 60)
    print(f"Result {i + 1}")
    print(f"Page: {metadata['page']}")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print(results["documents"][0][i][:500])
    print()


------------------------------------------------------------
Result 1
Page: 16
Distance: 0.3177
deep and reaching half down the hall.
After a time she heard a little pattering of
feet in the distance, and she hastily dried her
eyes to see what was coming. It was the White
Rabbit returning, splendidly dressed, with a pair
of white kid gloves in one hand and a large
fan in the other : he came trotting along in a
great hurry, muttering to himself as he came,
“Oh ! the Duchess, the Duchess ! Oh ! won’t she
be savage if I ’ve kept her waiting !” Alice
felt so desperate that she was ready to ask

------------------------------------------------------------
Result 2
Page: 9
Distance: 0.3263
curred to her that she ought to have wondered
at this, but at the time it all seemed quite
natural) ; but when the Rabbit actually took a
watch out of its waistcoat-pocket, and looked at
it, and then hurried on, Alice started to her
feet, for it ﬂashed across her mind that she had
never before seen a rabbi

In [ ]:

import json
import hashlib
from pathlib import Path


def summarize_book(
    collection,
    book_id,
    book_title,
    llm,
    cache_dir="../summary_cache",
    chunks_per_map=2,
    reduce_batch_size=8
):
    """
    Summarize an entire book using Map-Reduce.

    - Reads all chunks for the book from ChromaDB.
    - Caches intermediate summaries.
    - Reuses the final summary when the book and settings match.
    - Resumes cached mapping work if interrupted.
    """

    cache_path = Path(cache_dir)
    cache_path.mkdir(parents=True, exist_ok=True)

    # 1. Load every chunk for this book from ChromaDB
    stored = collection.get(
        where={"book_id": book_id},
        include=["documents", "metadatas"]
    )

    if not stored["documents"]:
        raise ValueError(f"No chunks found for book_id: {book_id}")

    # Restore the original book order
    book_chunks = sorted(
        zip(stored["documents"], stored["metadatas"]),
        key=lambda item: item[1]["chunk_index"]
    )

    texts = [item[0] for item in book_chunks]

    # 2. Create a fingerprint to detect book-content changes
    fingerprint = hashlib.sha256(
        "\n".join(texts).encode("utf-8")
    ).hexdigest()

    safe_book_id = "".join(
        c if c.isalnum() or c in "-_" else "_"
        for c in book_id
    )

    map_cache_file = cache_path / f"{safe_book_id}_map.json"
    final_cache_file = cache_path / f"{safe_book_id}_summary.json"

    # 3. Reuse the final summary if the source is unchanged
    if final_cache_file.exists():
        cached = json.loads(
            final_cache_file.read_text(encoding="utf-8")
        )

        if (
            cached.get("fingerprint") == fingerprint
            and cached.get("chunks_per_map") == chunks_per_map
            and cached.get("reduce_batch_size") == reduce_batch_size
        ):
            print("Using cached book summary.")
            return cached["summary"]

    # 4. Load intermediate summaries if they match this book
    mapped_summaries = []

    if map_cache_file.exists():
        cached_map = json.loads(
            map_cache_file.read_text(encoding="utf-8")
        )

        if (
            cached_map.get("fingerprint") == fingerprint
            and cached_map.get("chunks_per_map") == chunks_per_map
        ):
            mapped_summaries = cached_map["summaries"]

    # 5. MAP: summarize groups of chunks
    for start in range(
        len(mapped_summaries) * chunks_per_map,
        len(texts),
        chunks_per_map
    ):
        batch = texts[start:start + chunks_per_map]

        prompt = f"""
You are summarizing a section of the book "{book_title}".

Summarize only the supplied text. Preserve important events,
characters, actions, motivations, and cause-and-effect relationships.
Keep the summary concise but informative. Do not invent details.
Maintain the order of events and do not add an introduction.

BOOK TEXT:
{"".join(f"\\n\\nPart {i + 1}:\\n{text}" for i, text in enumerate(batch))}

SECTION SUMMARY:
"""

        response = llm.invoke(prompt)
        mapped_summaries.append(response.content.strip())

        # Save progress after every map operation
        map_cache_file.write_text(
            json.dumps({
                "fingerprint": fingerprint,
                "chunks_per_map": chunks_per_map,
                "summaries": mapped_summaries
            }, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        print(
            f"Mapped {min(start + chunks_per_map, len(texts))}"
            f"/{len(texts)} chunks"
        )

    # 6. REDUCE: combine summaries until one remains
    current_summaries = mapped_summaries

    while len(current_summaries) > 1:
        next_summaries = []

        for start in range(
            0, len(current_summaries), reduce_batch_size
        ):
            batch = current_summaries[
                start:start + reduce_batch_size
            ]

            if len(batch) == 1:
                next_summaries.append(batch[0])
                continue

            is_final = len(current_summaries) <= reduce_batch_size

            if is_final:
                instruction = f"""
Create a coherent, comprehensive summary of the entire book
"{book_title}". Cover the major events, characters, their motivations,
important developments, and the ending. Preserve chronological flow,
remove repetition, and do not invent facts. Use clear headings where
helpful. Target approximately 800-1200 words.
"""
            else:
                instruction = f"""
Combine these summaries into one shorter, coherent summary.
Preserve important events, characters, and cause-and-effect details.
Remove repetition, maintain the order of events, and do not invent facts.
"""

            prompt = f"""
{instruction}

SUMMARIES:
{"".join(f"\\n\\nSummary {i + 1}:\\n{s}" for i, s in enumerate(batch))}

COMBINED SUMMARY:
"""

            response = llm.invoke(prompt)
            next_summaries.append(response.content.strip())

        current_summaries = next_summaries

    final_summary = current_summaries[0]

    # 7. Cache the completed summary
    final_cache_file.write_text(
        json.dumps({
            "book_id": book_id,
            "book_title": book_title,
            "fingerprint": fingerprint,
            "chunks_per_map": chunks_per_map,
            "reduce_batch_size": reduce_batch_size,
            "summary": final_summary
        }, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("Book summary generated and cached.")
    return final_summary


In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv("../.env")

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=400,
    api_key=api_key
)

print("LLM initialized successfully.")

LLM initialized successfully.


In [ ]:
import time
from groq import RateLimitError

In [10]:

summary = summarize_book(
    collection=alice_collection,
    book_id="alice_wonderland",
    book_title="Alice's Adventures in Wonderland",
    llm=llm
)

print(summary)


Mapped 4/245 chunks
Mapped 8/245 chunks
Mapped 12/245 chunks
Mapped 16/245 chunks
Mapped 20/245 chunks
Mapped 24/245 chunks
Mapped 28/245 chunks
Mapped 32/245 chunks
Mapped 36/245 chunks
Mapped 40/245 chunks
Mapped 44/245 chunks
Mapped 48/245 chunks
Mapped 52/245 chunks
Mapped 56/245 chunks
Mapped 60/245 chunks
Mapped 64/245 chunks
Mapped 68/245 chunks
Mapped 72/245 chunks
Mapped 76/245 chunks
Mapped 80/245 chunks
Mapped 84/245 chunks


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m557s6myera9dyghb3d72pye` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6073, Requested 1959. Please try again in 240ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}